In [1]:
import pandas as pd
import numpy as np


def extract_sample_group(run_id: str) -> str:
    """
    Extrahiert eine grobe Gruppen-/Familien-ID aus run_id.
    Beispiele:
      ebpfangel_run_r001_rep001_... -> ebpfangel
      jimmy_r001_rep001_...         -> jimmy
      quema100_r001_rep001_...      -> quema100
      copy_rep001_...               -> copy
      fio_rep001_...                -> fio
      compress_rep001_...           -> compress
    """
    run_id = str(run_id)

    if "_run_r" in run_id:
        return run_id.split("_run_r")[0]
    if "_r" in run_id and "_rep" in run_id:
        return run_id.split("_r")[0]
    if "_rep" in run_id:
        return run_id.split("_rep")[0]

    return run_id


def safe_div(numerator: float, denominator: float) -> float:
    return numerator / denominator if denominator else float("nan")


def load_detection_log(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path, dtype=str)

    required_cols = {
        "run_id",
        "label",
        "attack_start_time",
        "user_created_time",
        "detection_status",
        "detection_time",
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Fehlende Spalten in CSV: {sorted(missing)}")

    # Normalisieren
    df["label"] = df["label"].str.strip().str.lower()
    df["detection_status"] = pd.to_numeric(df["detection_status"], errors="coerce").fillna(0).astype(int)

    # Zeitstempel parsen
    df["attack_start_time"] = pd.to_datetime(df["attack_start_time"], utc=True, errors="coerce")
    df["user_created_time"] = pd.to_datetime(df["user_created_time"], utc=True, errors="coerce")
    df["detection_time"] = pd.to_datetime(
        df["detection_time"].replace({"-": pd.NA, "": pd.NA}),
        utc=True,
        errors="coerce",
    )

    # Klassen und Vorhersagen
    df["is_positive"] = df["label"].eq("ransomware")
    df["pred_positive"] = df["detection_status"].eq(1)

    # Startzeit für Latenz: primär attack_start_time, fallback user_created_time
    df["event_start_time"] = df["attack_start_time"].fillna(df["user_created_time"])

    # Detektionslatenz in Sekunden
    df["detection_delay_sec"] = (df["detection_time"] - df["event_start_time"]).dt.total_seconds()

    # Nur für erkannte positive Fälle sinnvoll
    df.loc[~(df["is_positive"] & df["pred_positive"]), "detection_delay_sec"] = np.nan

    # Für Familien-/Workload-Sicht
    df["sample_group"] = df["run_id"].map(extract_sample_group)

    return df


def evaluate_detection(df: pd.DataFrame) -> dict:
    tp = int((df["is_positive"] & df["pred_positive"]).sum())
    fn = int((df["is_positive"] & ~df["pred_positive"]).sum())
    fp = int((~df["is_positive"] & df["pred_positive"]).sum())
    tn = int((~df["is_positive"] & ~df["pred_positive"]).sum())

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)              # TPR / Sensitivity
    specificity = safe_div(tn, tn + fp)         # TNR
    fpr = safe_div(fp, fp + tn)
    fnr = safe_div(fn, fn + tp)
    accuracy = safe_div(tp + tn, len(df))
    f1 = (
        2 * precision * recall / (precision + recall)
        if not np.isnan(precision) and not np.isnan(recall) and (precision + recall) > 0
        else float("nan")
    )

    tp_delays = df.loc[df["is_positive"] & df["pred_positive"], "detection_delay_sec"].dropna()

    if len(tp_delays) > 0:
        latency_stats = {
            "count": int(tp_delays.shape[0]),
            "mean_sec": float(tp_delays.mean()),
            "median_sec": float(tp_delays.median()),
            "min_sec": float(tp_delays.min()),
            "max_sec": float(tp_delays.max()),
            "std_sec": float(tp_delays.std(ddof=1)) if len(tp_delays) > 1 else 0.0,
        }
    else:
        latency_stats = {
            "count": 0,
            "mean_sec": float("nan"),
            "median_sec": float("nan"),
            "min_sec": float("nan"),
            "max_sec": float("nan"),
            "std_sec": float("nan"),
        }

    latency_by_group = (
        df.loc[df["is_positive"] & df["pred_positive"], ["sample_group", "detection_delay_sec"]]
          .groupby("sample_group")["detection_delay_sec"]
          .agg(["count", "mean", "median", "min", "max", "std"])
          .reset_index()
          .sort_values("sample_group")
    )

    confusion_df = pd.DataFrame(
        {
            "Actual ransomware": [tp, fn],
            "Actual benign": [fp, tn],
        },
        index=["Predicted ransomware", "Predicted benign"]
    )

    metrics_df = pd.DataFrame(
        [
            ("TP", tp),
            ("FP", fp),
            ("TN", tn),
            ("FN", fn),
            ("Precision", precision),
            ("Recall / TPR", recall),
            ("Specificity / TNR", specificity),
            ("FPR", fpr),
            ("FNR", fnr),
            ("F1", f1),
            ("Accuracy", accuracy),
        ],
        columns=["Metric", "Value"]
    )

    latency_df = pd.DataFrame(
        [
            ("Detected ransomware runs", latency_stats["count"]),
            ("Mean detection time (s)", latency_stats["mean_sec"]),
            ("Median detection time (s)", latency_stats["median_sec"]),
            ("Min detection time (s)", latency_stats["min_sec"]),
            ("Max detection time (s)", latency_stats["max_sec"]),
            ("Std detection time (s)", latency_stats["std_sec"]),
        ],
        columns=["Metric", "Value"]
    )

    return {
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "confusion_df": confusion_df,
        "metrics_df": metrics_df,
        "latency_df": latency_df,
        "latency_by_group_df": latency_by_group,
        "evaluated_df": df.copy(),
    }


def print_detection_report(results: dict) -> None:
    tp = results["tp"]
    fp = results["fp"]
    tn = results["tn"]
    fn = results["fn"]

    print("=" * 80)
    print("DETECTION EVALUATION REPORT")
    print("=" * 80)
    print(f"TP = {tp}")
    print(f"FP = {fp}")
    print(f"TN = {tn}")
    print(f"FN = {fn}")

    print("\nConfusion Matrix:")
    print(results["confusion_df"])

    print("\nClassification Metrics:")
    print(results["metrics_df"].to_string(index=False))

    print("\nDetection Latency:")
    print(results["latency_df"].to_string(index=False))

    if not results["latency_by_group_df"].empty:
        print("\nDetection Latency by Ransomware Group:")
        print(results["latency_by_group_df"].to_string(index=False))


# ============================================================
# AUSFÜHRUNG IM NOTEBOOK
# ============================================================

FILE_PATH = "./detection_log.csv"

df = load_detection_log(FILE_PATH)
results = evaluate_detection(df)
print_detection_report(results)

# Optional: DataFrames direkt anschauen
#display(results["metrics_df"])
#display(results["latency_df"])
#display(results["latency_by_group_df"])
#display(results["evaluated_df"])

DETECTION EVALUATION REPORT
TP = 15
FP = 0
TN = 15
FN = 0

Confusion Matrix:
                      Actual ransomware  Actual benign
Predicted ransomware                 15              0
Predicted benign                      0             15

Classification Metrics:
           Metric  Value
               TP   15.0
               FP    0.0
               TN   15.0
               FN    0.0
        Precision    1.0
     Recall / TPR    1.0
Specificity / TNR    1.0
              FPR    0.0
              FNR    0.0
               F1    1.0
         Accuracy    1.0

Detection Latency:
                   Metric     Value
 Detected ransomware runs 15.000000
  Mean detection time (s) 52.553618
Median detection time (s) 60.459232
   Min detection time (s) 32.187997
   Max detection time (s) 74.560113
   Std detection time (s) 14.307470

Detection Latency by Ransomware Group:
sample_group  count      mean    median       min       max      std
   ebpfangel      5 33.751404 32.466635 32.187997 39